# Text 분류 — 시험용 완성 코드

위에서부터 순서대로 실행. **★ 설정**만 데이터에 맞게 바꾸면 됩니다.
label 확인·수정은 **셀을 나눠** 처리합니다. (이진/다중 구분 없이 동작 · 라벨=예측=제출값)

> 변수명은 **이 파일 단독 실행** 기준(plain)입니다. 시험은 **한 커널**에서 3문제를 이어 풀어 `X`·`best`·`sub` 같은 변수가 서로 덮어써집니다 — 정형=`p1_`, 텍스트=`p2_`, 이미지=`p3_` 접두사로 문제별로 분리하세요.

## 1. 설정 · 로드  ← ★ 여기만 수정

In [1]:
import pandas as pd, numpy as np, re, joblib
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 42
RESULT_2 = '01012345678_2'             # 휴대폰번호_문제번호 (하이픈·공백 없이). Text=2

# ▼ 데이터 선택 (하나만) — '파일 여는 것'만. 문장/라벨 컬럼은 데이터 본 뒤 '3.' 에서
TRAIN_URL, TEST_URL = 'https://github.com/gzone2000/TEMP_TEST/raw/refs/heads/master/A_comment_train.xlsx', 'https://github.com/gzone2000/TEMP_TEST/raw/refs/heads/master/A_comment_test.xlsx'  # [1] A댓글·이진·train/test 분리
# TRAIN_URL, TEST_URL = 'https://github.com/gzone2000/TEMP_TEST/raw/refs/heads/master/ChatbotData.csv', None       # [2] 챗봇·다중(3)
# TRAIN_URL, TEST_URL = 'https://github.com/gzone2000/TEMP_TEST/raw/refs/heads/master/IMDb_small_data.csv', None    # [3] IMDb·이진(영어)

def read_any(u):
    return pd.read_excel(u) if str(u).lower().endswith(('.xlsx', '.xls')) else pd.read_csv(u)

df_all = read_any(TRAIN_URL)                        # 원본(불변) — 분할은 컬럼 정한 뒤 '3.' 에서
df_test_raw = read_any(TEST_URL) if TEST_URL else None
for _d in [df_all] + ([df_test_raw] if TEST_URL else []):   # 저장 시 생긴 인덱스 컬럼 제거
    _d.drop(columns=[c for c in _d.columns if str(c).startswith('Unnamed')], inplace=True, errors='ignore')
print('불러온 데이터:', df_all.shape, '| test 파일:', df_test_raw.shape if TEST_URL else '없음(train 에서 분리)')
df_all.head()

불러온 데이터: (251, 2) | test 파일: (101, 2)


,data,label
0,재미는 있는데 시간이 짧은게 아쉽네요~,긍정
1,"OO 관련 내용은 우리 직원과는 거리가 멀었음, 특히, 사내에 홍보할 내용은 아니라고 봄",부정
2,스토리가 너무 딱딱해서 별로였음,부정
3,프로그램A 화이팅하세요!!,긍정
4,높은 곳에 올라가는 모습이 너무 위험해 보여요.,부정


## 2. 데이터 확인  (어느 컬럼이 문장이고 어느 컬럼이 라벨인지 본다)

In [2]:
print('컬럼:', list(df_all.columns))
print(); print('행 예시:'); print(df_all.head(3))

컬럼: ['data', 'label']

행 예시:
                                                data label
0                              재미는 있는데 시간이 짧은게 아쉽네요~    긍정
1  OO 관련 내용은 우리 직원과는 거리가 멀었음, 특히, 사내에 홍보할 내용은 아니라고 봄    부정
2                                  스토리가 너무 딱딱해서 별로였음    부정


## 3. 문장 · 라벨 컬럼 지정 · 분할  ← ★ 위 '데이터 확인'을 보고 채운다

In [3]:
# ★ 위 컬럼에서 '문장 컬럼(TEXT_COL)'과 '라벨 컬럼(TARGET)'을 직접 지정
TEXT_COL = 'data'            # [1] A댓글:'data' · [2] 챗봇:'Q' · [3] IMDb:'review'
TARGET   = 'label'           # [1] 'label' · [2] 'label' · [3] 'sentiment'
PRED_COL = TARGET            # 제출 결과 컬럼명 — 지문이 다른 이름을 지정하면 그 이름으로

print('텍스트 예시:', df_all[TEXT_COL].iloc[0])
print('label 고유값:', sorted(df_all[TARGET].dropna().unique().tolist(), key=str))   # 숫자면 '4.' 에서 매핑
print(df_all[TARGET].value_counts())

# test 가 따로 없으면 여기서 분리 (stratify=라벨). df_all 은 원본이라 재실행해도 안전
if df_test_raw is None:
    df, df_test = train_test_split(df_all, test_size=0.2, random_state=SEED, stratify=df_all[TARGET])
else:
    df, df_test = df_all.copy(), df_test_raw.copy()
df, df_test = df.reset_index(drop=True), df_test.reset_index(drop=True)
df_test_orig = df_test.copy()          # 제출은 test 원본 구조로
print('Train', df.shape, '/ Test', df_test.shape)

텍스트 예시: 재미는 있는데 시간이 짧은게 아쉽네요~
label 고유값: ['긍정', '부정']
label
긍정    126
부정    125
Name: count, dtype: int64
Train (251, 2) / Test (101, 2)


## 4. 라벨 값 수정  ← ★ 숫자일 때만 채운다

In [4]:
# 제출값 = label 그대로. 라벨이 '이미 문자'면 None, '숫자'면 의미대로 매핑.
LABEL_MAP = None
# 예) 챗봇(0/1/2): LABEL_MAP = {0: '일상', 1: '부정', 2: '긍정'}
# 예) IMDb(0/1):   LABEL_MAP = {0: 'negative', 1: 'positive'}

if LABEL_MAP:
    df[TARGET] = df[TARGET].map(LABEL_MAP)
    if TARGET in df_test.columns:
        df_test[TARGET] = df_test[TARGET].map(LABEL_MAP)
print('최종 label:', sorted(df[TARGET].unique().tolist(), key=str))

최종 label: ['긍정', '부정']


## 5. 전처리  (정제 → train/val 분할)

In [5]:
def clean(t):
    t = re.sub('[^가-힣a-zA-Z0-9 ]', ' ', str(t))       # 특수문자 제거
    return ' '.join(t.split())                          # 다중 공백 정리

df[TEXT_COL] = df[TEXT_COL].map(clean)
df_test[TEXT_COL] = df_test[TEXT_COL].map(clean)
df = df[df[TEXT_COL] != ''].reset_index(drop=True)

has_y_test = TARGET in df_test.columns
Xtr, Xva, ytr, yva = train_test_split(
    df[TEXT_COL], df[TARGET], test_size=0.2, random_state=SEED, stratify=df[TARGET])
print('train', len(Xtr), '/ val', len(Xva))

train 200 / val 51


## 6. TF-IDF · 모델 비교 · 평가

In [6]:
# 한국어 짧은 문장엔 문자 n-gram(char_wb)이 공백 토큰화보다 강하다
vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), max_features=50000, min_df=2, sublinear_tf=True)
Xtr_v = vec.fit_transform(Xtr); Xva_v = vec.transform(Xva)

models = {'LinearSVC': LinearSVC(C=1), 'LogReg': LogisticRegression(max_iter=1000), 'NB': MultinomialNB(alpha=0.1)}
cv = {n: cross_val_score(m, Xtr_v, ytr, cv=5).mean() for n, m in models.items()}
best_name = max(cv, key=cv.get)
best = models[best_name].fit(Xtr_v, ytr)
print('CV:', {n: round(s, 4) for n, s in cv.items()}, '→ 1위', best_name)

yva_pred = best.predict(Xva_v)
print(f'Val Acc {accuracy_score(yva, yva_pred):.4f} / F1(macro) {f1_score(yva, yva_pred, average="macro"):.4f}')
print(classification_report(yva, yva_pred))

CV: {'LinearSVC': np.float64(0.89), 'LogReg': np.float64(0.915), 'NB': np.float64(0.925)} → 1위 NB
Val Acc 0.9020 / F1(macro) 0.9020
              precision    recall  f1-score   support

          긍정       0.92      0.88      0.90        26
          부정       0.88      0.92      0.90        25

    accuracy                           0.90        51
   macro avg       0.90      0.90      0.90        51
weighted avg       0.90      0.90      0.90        51



## 7. 결과 만들고 ⭐ 눈으로 확인

> 원문과 예측 라벨을 나란히 봅니다. 명백히 반대로 찍힌 게 있으면 **저장 전에** 원인부터.

In [7]:
pred = best.predict(vec.transform(df_test[TEXT_COL]))

sub = df_test_orig.drop(columns=[c for c in [TARGET] if c in df_test_orig.columns]).copy()
sub[PRED_COL] = pred

rng = np.random.RandomState(SEED)                       # ★ 원문 vs 예측 눈 확인
for i in rng.choice(len(sub), min(8, len(sub)), replace=False):
    print(f'[{sub[PRED_COL].iloc[i]}]  {str(df_test_orig[TEXT_COL].iloc[i])[:50]}')
if has_y_test:
    print('\nTest Acc', round(accuracy_score(df_test[TARGET], pred), 4))
sub.head()

[긍정]  사내방송 이제 바다에서도 맘껏 사용할수 있네요
[부정]  다양한 주제에 대해서 다뤄주세요
[긍정]  달려온 길보다 달려갈 길이 더욱 기대됩니다. 축하 축하~
[부정]  지루한 구성 재미없어요
[부정]  영상 초반 이미지로 구성한 부분이 눈에 안들어오네요
[부정]  바다에 관련된 정보만 있는게 편향되게 느껴졌어요
[긍정]  리얼 공감 에피소드네요. 연속으로 재미있게 보고 있습니다. 계속 연재 부탁드릴게요. 
[긍정]  항상 직원들과 함께 하는 방송되주세요. 다음화도 기대하겠습니다! 

Test Acc 0.8812


,data,label
0,유익한 정보 감사합니다. 취미로 바다에 자주 나가는 편인데 이런 기술이 있는지 몰랐네요.,긍정
1,발빠른 영입 수고 했네요 고객들이 좋아합니다.,긍정
2,프로그램C 주제가 우리의 기술력이 바탕이 된다는점~~..항상 응원합니다,긍정
3,언제나 밝은 미소로 열심히 근무하시는 청원 경찰 분들께 감사드립니다. 항상 건강하시...,긍정
4,안전사고에 대한 개념도 함께 소개 되었으면 더 완벽했을 것 같네요,부정


## 8. ⭐ csv · 모델 저장

> 텍스트는 예측에 벡터라이저가 필요하므로 **모델+벡터라이저 함께** 저장.

In [8]:
import os
CSV, MODEL = f'{RESULT_2}.csv', f'{RESULT_2}.pkl'
assert len(sub) == len(df_test_orig), '행 수 불일치'      # ★ 저장 전에 검증 (0점 나는 두 원인)
assert sub[PRED_COL].isnull().sum() == 0, '예측 결측'
sub.to_csv(CSV, index=False)      # 결과 csv
joblib.dump({'model': best, 'vectorizer': vec}, MODEL)  # 모델 + 벡터라이저
print('저장 완료:', [f for f in os.listdir('.') if f.startswith(RESULT_2)])

저장 완료: ['01012345678_2.pkl', '01012345678_2.csv']
